# Softmax 순전파와 역전파

Triton의 순전파와 역전파를 이해하기 위한 비효율적인 softmax 구현입니다. 순전파와 역전파 모두에서 한 행 전체를 하나의 블록에 넣어 처리합니다.

[Triton의 LayerNorm](https://triton-lang.org/main/getting-started/tutorials/05-layer-norm.html) 및 [Softmax](https://triton-lang.org/main/getting-started/tutorials/02-fused-softmax.html) 튜토리얼을 참고해 작성했습니다.

먼저 순전파 커널을 정의합니다. Softmax는 다음과 같습니다. $\sigma(z_i) = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}$.

1. 한 행의 데이터를 불러오고 마스크를 정의합니다.
2. 수치 안정성을 위해 행의 최댓값을 각 요소에서 뺍니다. 이렇게 하면 $e^{z_i}$를 계산할 때 오버플로를 방지할 수 있습니다.
3. Softmax 식을 구현합니다. 각 값의 지수 함수를 계산한 뒤 그 합으로 나눕니다.

In [ ]:
import torch
from torch.testing import assert_close
import torch.nn.functional as F

import triton
import triton.language as tl


@triton.jit
def softmax_fwd_kernel(
    x_ptr,
    output_ptr,
    x_rows,
    x_cols,
    BLOCK_SIZE:tl.constexpr
):
    pid = tl.program_id(axis=0) # 현재 행의 ID
    
    row_start = pid * x_cols
    row_offset = row_start + tl.arange(0, BLOCK_SIZE)
    
    mask = row_offset < row_start + x_cols
    
    row = tl.load(x_ptr + row_offset, mask=mask, other=-float('inf'))
    
    row = row - tl.max(row, axis=0) # 수치 안정성을 위해 최댓값을 뺌
    num = tl.exp(row) # 각 요소의 지수 함수 계산
    denom = tl.sum(num, axis=0) # 행의 요소를 합산
    output = num / denom # Softmax 결과
    
    tl.store(output_ptr + row_offset, output, mask=mask)

이제 순전파를 수행하는 Python 함수를 정의합니다. 각 행을 처리하려면 해당 행의 모든 요소를 포함하는 블록이 필요하지만, Triton 블록 크기는 2의 거듭제곱이어야 합니다.

행을 따라가는 1차원 실행 그리드를 만듭니다.

In [ ]:
def softmax_fwd(x):
    output = torch.empty_like(x)
    x_rows, x_cols = x.shape
    
    # 블록 크기 설정: Triton은 2의 거듭제곱 크기만 지원
    BLOCK_SIZE = triton.next_power_of_2(x_cols)
    # 1차원 실행 그리드
    grid = lambda meta: (x_rows,)
    
    softmax_fwd_kernel[grid](x, output, x_rows, x_cols, BLOCK_SIZE=BLOCK_SIZE)
    
    return output

`assert_close`를 사용해 Triton의 softmax 커널 결과를 PyTorch 구현과 비교합니다.

In [3]:
torch.manual_seed(0)
x = torch.rand((10, 6), device='cuda')

output_triton = softmax_fwd(x)

In [4]:
x.requires_grad = True
output_torch = F.softmax(x, dim=-1)

assert_close(output_triton, output_torch)

앞에서 `x`에 `requires_grad`를 설정했습니다. 이 설정으로 PyTorch 텐서의 역전파를 실행할 수 있습니다. 임의의 원-핫 인코딩 타깃 `y`를 만들고 [`CrossEntropyLoss`](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)를 사용해 `x`에 대한 그래디언트를 구합니다.

In [ ]:
# 임의의 타깃 준비
y = torch.zeros_like(x)
inds = (torch.arange(0, y.shape[0]), torch.randint(0, 3, (y.shape[0],)))
y[inds] = 1

# 손실을 정의하고 역전파 실행
loss_fn = torch.nn.CrossEntropyLoss()
loss = loss_fn(output_torch, y)
loss.backward()

# 나중에 사용할 그래디언트 텐서 저장
torch_xgrad = x.grad.detach().clone()

이제 PyTorch와 비교하기 위해 softmax 역전파 커널을 작성합니다. 학습용으로만 만든 매우 비효율적인 구현입니다. Softmax를 계산 그래프로 나누고 그래프를 따라 단계별로 역전파를 수행합니다.

다음은 softmax의 순전파 계산 그래프입니다.

![계산 그래프](images/comp_graph.png)

1. 역전파 계산에 필요한 값을 다시 생성하기 위해 순전파를 실행합니다.
2. 단계별로 그래디언트를 계산합니다(주석 참고).
3. 그래디언트를 저장합니다.

In [ ]:
@triton.jit
def softmax_bwd_kernel(
    x_ptr,
    dy_ptr,
    dx_ptr,
    x_rows,
    x_cols,
    BLOCK_SIZE: tl.constexpr
):
    pid = tl.program_id(axis=0) # 현재 행의 ID
    
    row_start = pid * x_cols # Softmax를 적용한 행의 시작 위치
    row_offset = row_start + tl.arange(0, BLOCK_SIZE)
    
    mask = row_offset < row_start + x_cols
    
    # x 행과 입력 그래디언트 행 불러오기
    x_row = tl.load(x_ptr + row_offset, mask=mask, other=-float('inf'))
    dy_row = tl.load(dy_ptr + row_offset, mask=mask, other=0.0)
    
    x_normed = x_row - tl.max(x_row, axis=0) # 수치 안정성을 위해 최댓값을 뺌
    
    num = tl.exp(x_normed) # 각 요소의 지수 함수 계산
    denom = tl.sum(num, axis=0) # 행의 요소를 합산
    
    # 미분을 쉽게 이해하도록 식을 약간 변형
    inverted = 1 / denom # 분모의 역수
    output = num * inverted # 분자에 분모의 역수를 곱함
    
    # 마지막 softmax 순전파 단계의 두 구성 요소에 대한 그래디언트 계산
    num_grad = dy_row * inverted
    inverted_grad = dy_row * num
    inverted_grad = tl.sum(inverted_grad, axis=0)
    
    # softmax 분모에 대한 그래디언트
    denom_grad = (-1 / (denom * denom)) * inverted_grad
    
    # 분모(분자의 합)에서 온 그래디언트를 분자 그래디언트에 더함
    num_grad += tl.full(num_grad.shape, 1.0, dtype=tl.float32) * denom_grad
    
    # 정규화된 x 값에 대한 그래디언트 계산
    normed_grad = num_grad * tl.exp(x_normed)
    
    # 정규화 연산을 거치는 그래디언트 계산
    x_grad = normed_grad
    max_grad = -normed_grad
    max_grad = tl.sum(max_grad, axis=0)
    x_grad_2 = max_grad * tl.where(x_row == tl.max(x_row, axis=0), tl.full(x_grad.shape, 1.0, dtype=tl.float32), tl.zeros(x_grad.shape, dtype=tl.float32))
   
    tl.store(dx_ptr + row_offset, x_grad + x_grad_2, mask=mask)

이제 사용자 정의 순전파와 역전파를 구현할 수 있도록 PyTorch autograd 함수를 만듭니다.

In [ ]:
class Softmax(torch.autograd.Function):

    @staticmethod
    def forward(ctx, x):
        output = torch.empty_like(x)
        x_rows, x_cols = x.shape

        BLOCK_SIZE = triton.next_power_of_2(x_cols)
        grid = lambda meta: (x_rows,)

        softmax_fwd_kernel[grid](x, output, x_rows, x_cols, BLOCK_SIZE=BLOCK_SIZE)
        
        ctx.save_for_backward(x) # 역전파에서 사용할 x를 캐시에 저장
        return output

    @staticmethod
    def backward(ctx, dy):
        x = ctx.saved_tensors[0] # 캐시에서 x 가져오기
        dx = torch.empty_like(x)
        x_rows, x_cols = x.shape

        BLOCK_SIZE = triton.next_power_of_2(x_cols)
        grid = lambda meta: (x_rows,)

        softmax_bwd_kernel[grid](x, dy, dx, x_rows, x_cols, BLOCK_SIZE=BLOCK_SIZE)

        return dx


softmax = Softmax.apply

순전파 결과를 비교해 구현을 테스트합니다.

In [ ]:
x.grad.data.zero_() # x의 그래디언트를 0으로 초기화

output_triton2 = softmax(x)

assert_close(output_triton, output_triton2)

이제 역전파 결과를 확인합니다.

In [10]:
triton_loss = loss_fn(output_triton2, y)
triton_loss.backward()
assert_close(x.grad, torch_xgrad)

tensor([[ 0.0160,  0.0163, -0.0846,  0.0176,  0.0176,  0.0171],
        [-0.0835,  0.0178,  0.0160,  0.0182,  0.0158,  0.0157],
        [ 0.0162,  0.0180, -0.0842,  0.0158,  0.0177,  0.0163],
        [-0.0838,  0.0166,  0.0176,  0.0159,  0.0174,  0.0163],
        [ 0.0158, -0.0831,  0.0158,  0.0163,  0.0182,  0.0170],
        [-0.0840,  0.0172,  0.0174,  0.0181,  0.0156,  0.0157],
        [ 0.0169, -0.0823,  0.0160,  0.0157,  0.0171,  0.0166],
        [ 0.0161, -0.0825,  0.0186,  0.0158,  0.0163,  0.0158],
        [ 0.0171, -0.0833,  0.0155,  0.0176,  0.0166,  0.0165],
        [-0.0842,  0.0161,  0.0167,  0.0178,  0.0178,  0.0159]],
       device='cuda:0')
